# model

> Wav2vec 2.0 with a CTC head.
> 
> `build` constructs the model on CPU. `compile_model` traces it for one
> batch size and one clip length, the same way solid_lens compiles a network
> for one image size. The result is two tensors: CTC logits per frame, and
> the final hidden states.
>


In [ ]:
#| default_exp model


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np
from max.driver import CPU
from max.dtype import DType
from max.experimental.nn import Linear, Module
from max.experimental.tensor import Tensor, default_device
from max.graph import TensorType

from nada.audio import Wav2Vec2Model
from nada.config import Wav2Vec2Config, base_960h_config
from nada.core import repo_root
from nada.features import ctc_decode, load_vocab, prepare, read_wav
from nada.weights import assign_parameters, load_arrays


In [ ]:
#| export
class Wav2Vec2ForCTC(Module):
    """`Wav2Vec2Model` and a linear head from hidden states to character logits.

    Module names match Hugging Face, so a state dict loads without renaming.
    """

    def __init__(self, config: Wav2Vec2Config | None = None) -> None:
        self.config = config or base_960h_config()
        with default_device(CPU()):
            self.wav2vec2 = Wav2Vec2Model(self.config)
            self.lm_head = Linear(self.config.hidden_size, self.config.vocab_size)

    def forward(self, input_values: Tensor) -> tuple[Tensor, Tensor]:
        hidden = self.wav2vec2(input_values)
        return self.lm_head(hidden), hidden


In [ ]:
#| export
def build(config: Wav2Vec2Config | None = None) -> Wav2Vec2ForCTC:
    """Construct the model on CPU."""
    return Wav2Vec2ForCTC(config)


In [ ]:
#| export
def compile_model(model: Wav2Vec2ForCTC, *, batch: int, num_samples: int):
    """Compile the forward pass for `batch` clips of exactly `num_samples` samples.

    Frame count follows from the config. Recompile when the clip length changes.
    """
    model.config.output_length(num_samples)
    return model.compile(TensorType(DType.float32, [batch, num_samples], device=model.device))


In [ ]:
#| export
def load_checkpoint(model: Wav2Vec2ForCTC, path=None) -> None:
    """Load `models/wav2vec2.safetensors`, or `path` when given."""
    checkpoint = path or (repo_root() / "models" / "wav2vec2.safetensors")
    assign_parameters(model, load_arrays(checkpoint))


In [ ]:
#| export
def to_device(model: Wav2Vec2ForCTC, array: np.ndarray) -> Tensor:
    """Put `array` where the model lives. The default device may be a GPU the model is not on."""
    with default_device(model.device):
        return Tensor(np.ascontiguousarray(array))


`transcribe` ties the pieces together: normalize and pad, run the compiled graph, take the argmax per frame, and decode.


In [ ]:
#| export
def transcribe(model: Wav2Vec2ForCTC, compiled, waveforms, vocab: dict[int, str] | None = None) -> list[str]:
    """Text for each clip. `compiled` must have been built for this batch size and padded length."""
    values = prepare(waveforms, model.config)
    logits, _ = compiled(to_device(model, values))
    ids = np.asarray(logits.to_numpy()).argmax(axis=-1)
    return ctc_decode(ids, vocab or load_vocab())


In [ ]:
#| export
def _main() -> None:
    import sys

    if len(sys.argv) < 2:
        raise SystemExit("usage: python -m nada.model clip.wav [more.wav ...]")
    clips = [read_wav(path) for path in sys.argv[1:]]
    model = build()
    load_checkpoint(model)
    longest = max(clip.size for clip in clips)
    compiled = compile_model(model, batch=len(clips), num_samples=longest)
    for path, text in zip(sys.argv[1:], transcribe(model, compiled, clips), strict=True):
        print(f"{path}: {text}")


In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    _main()
